# EP03 — CNN Shape Calculations
**Exam Relevance:** 2025 Q4 (6 marks — show your workings)

This is a calculation question. You need to track the spatial dimensions through each layer.

**The formula you must know:**
```
output_size = floor((input_size - kernel_size + 2*padding) / stride) + 1
```
- **Valid padding** = no padding (padding = 0)
- **Same padding** = padding that keeps spatial size the same when stride=1

---

## Solving the 2025 Q4 Exactly

```
Input:          [32, 32, 3]   (H=32, W=32, C=3)
Conv1:          8 filters, kernel=5, stride=2, valid padding
MaxPool1:       kernel=2, stride=2
Conv2:          16 filters, kernel=3, stride=1, same padding
MaxPool2:       kernel=2, stride=2
FC1:            ? inputs → 10 outputs
```

### Step-by-step calculation:

In [ ]:
import math

def conv_output(h, w, kernel, stride, padding=0):
    """Compute output spatial dimensions of a conv/pool layer."""
    h_out = math.floor((h - kernel + 2 * padding) / stride) + 1
    w_out = math.floor((w - kernel + 2 * padding) / stride) + 1
    return h_out, w_out

def same_padding(kernel):
    """Padding needed to keep spatial size same (stride=1 assumed)."""
    return kernel // 2

print("=" * 55)
print("2025 Q4 — Step-by-Step CNN Shape Calculation")
print("=" * 55)

h, w, c = 32, 32, 3
print(f"\nInput:       [{h} x {w} x {c}]")

# Conv1: 8 filters, kernel=5, stride=2, valid padding (padding=0)
h, w = conv_output(h, w, kernel=5, stride=2, padding=0)
c = 8
print(f"\nConv1 (k=5, s=2, valid):")
print(f"  formula: floor((32 - 5 + 0) / 2) + 1 = floor(27/2) + 1 = 13 + 1 = 14")
print(f"  Output:  [{h} x {w} x {c}]")

# MaxPool1: kernel=2, stride=2
h, w = conv_output(h, w, kernel=2, stride=2, padding=0)
print(f"\nMaxPool1 (k=2, s=2):")
print(f"  formula: floor((14 - 2) / 2) + 1 = floor(12/2) + 1 = 6 + 1 = 7")
print(f"  Output:  [{h} x {w} x {c}]")

# Conv2: 16 filters, kernel=3, stride=1, same padding
p = same_padding(3)  # same padding for kernel=3
h, w = conv_output(h, w, kernel=3, stride=1, padding=p)
c = 16
print(f"\nConv2 (k=3, s=1, same padding → p={p}):")
print(f"  formula: floor((7 - 3 + 2*1) / 1) + 1 = floor(6/1) + 1 = 7")
print(f"  Output:  [{h} x {w} x {c}]  (same padding keeps spatial dims!)")

# MaxPool2: kernel=2, stride=2
h, w = conv_output(h, w, kernel=2, stride=2, padding=0)
print(f"\nMaxPool2 (k=2, s=2):")
print(f"  formula: floor((7 - 2) / 2) + 1 = floor(5/2) + 1 = 2 + 1 = 3")
print(f"  Output:  [{h} x {w} x {c}]")

# Flatten for FC
fc_inputs = h * w * c
print(f"\nFlatten → FC inputs:")
print(f"  {h} x {w} x {c} = {fc_inputs}")
print(f"\n{'='*55}")
print(f"ANSWER: {fc_inputs} inputs to the fully connected layer")
print(f"That corresponds to option (iii) 144 from the exam")
print(f"{'='*55}")

## Verify with PyTorch

In [ ]:
import torch
import torch.nn as nn

# Build the exact CNN from the exam and pass a dummy input through
class ExamCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=5, stride=2, padding=0),   # Conv1: valid
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # Pool1
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1),   # Conv2: same
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # Pool2
        )
        # We'll determine FC input size dynamically
        self.classifier = nn.Linear(144, 10)  # answer from our calculation

    def forward(self, x):
        x = self.features(x)
        print(f"  Shape before flatten: {x.shape}")
        x = x.view(x.size(0), -1)
        print(f"  Shape after flatten:  {x.shape}")
        x = self.classifier(x)
        return x

model = ExamCNN()
dummy = torch.randn(1, 3, 32, 32)  # [batch=1, C=3, H=32, W=32]

print("Passing a [1, 3, 32, 32] tensor through the exam CNN:")
out = model(dummy)
print(f"  Final output shape: {out.shape}  (1 image, 10 classes)")
print()
print("The flatten gives 3 * 3 * 16 = 144 ✓")

## General Practice — Track Any Architecture

In [ ]:
def trace_cnn_shapes(input_shape, layers):
    """
    Trace shapes through a CNN.
    
    input_shape: (H, W, C)
    layers: list of dicts describing each layer
      {'type': 'conv', 'filters': N, 'kernel': K, 'stride': S, 'padding': 'valid'/'same'}
      {'type': 'pool', 'kernel': K, 'stride': S}
    """
    h, w, c = input_shape
    print(f"Input:  H={h}, W={w}, C={c}")
    
    for i, layer in enumerate(layers, 1):
        if layer['type'] == 'conv':
            k, s = layer['kernel'], layer['stride']
            if layer.get('padding') == 'same':
                p = k // 2
            else:
                p = 0
            h = math.floor((h - k + 2*p) / s) + 1
            w = math.floor((w - k + 2*p) / s) + 1
            c = layer['filters']
            pad_str = f'p={p} (same)' if layer.get('padding') == 'same' else 'valid'
            print(f"Conv{i}:  k={k}, s={s}, {pad_str} → H={h}, W={w}, C={c}")
        elif layer['type'] == 'pool':
            k, s = layer['kernel'], layer['stride']
            h = math.floor((h - k) / s) + 1
            w = math.floor((w - k) / s) + 1
            print(f"Pool{i}:  k={k}, s={s}         → H={h}, W={w}, C={c}")
    
    fc_inputs = h * w * c
    print(f"Flatten: {h} × {w} × {c} = {fc_inputs} FC inputs")
    return fc_inputs

# Reproduce the exam architecture
print("=== 2025 Exam Architecture ===")
exam_layers = [
    {'type': 'conv', 'filters': 8,  'kernel': 5, 'stride': 2, 'padding': 'valid'},
    {'type': 'pool', 'kernel': 2, 'stride': 2},
    {'type': 'conv', 'filters': 16, 'kernel': 3, 'stride': 1, 'padding': 'same'},
    {'type': 'pool', 'kernel': 2, 'stride': 2},
]
trace_cnn_shapes((32, 32, 3), exam_layers)

## Exam Quick-Reference Summary

**The formula:**
```
output = floor((input - kernel + 2*padding) / stride) + 1
```

**Padding shortcuts:**
- Valid padding → padding = 0
- Same padding → padding = kernel // 2 (keeps size when stride=1)

**FC input count:**
```
FC_inputs = final_H × final_W × final_channels
```

**2025 Answer walkthrough:**
```
[32,32,3] → Conv1(k=5,s=2,valid) → [14,14,8] → Pool(k=2,s=2) → [7,7,8]
         → Conv2(k=3,s=1,same)  → [7,7,16] → Pool(k=2,s=2) → [3,3,16]
         → Flatten → 3×3×16 = 144
```
Answer: **iii. 144**